In [240]:
import openai
# Core
import os
import re
import io
import logging
import pathlib
import sqlite3
from typing import TypedDict, List

# I/O and environment
from dotenv import load_dotenv, find_dotenv
from IPython.display import display, Markdown, Image

# LangChain
from langchain.chains import RetrievalQA, LLMChain, StuffDocumentsChain
from langchain_core.prompts import ChatPromptTemplate
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Pinecone
from pinecone import Pinecone
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# LangGraph
from langgraph.graph import START, StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver


In [241]:
# Config logging
logging.basicConfig(level=logging.INFO)

In [242]:
# Load enviroment
load_dotenv(find_dotenv())
memory = SqliteSaver(sqlite3.connect(":memory:", check_same_thread=False))
openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv('PINECONE_API_KEY')

# Set the api keys
openai.api_key =openai_api_key
client = openai.OpenAI()

MODEL_NAME = "gpt-4o-mini" 

In [243]:
# Inicializar OpenAIEmbeddings desde langchain
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)


pc = Pinecone(api_key=pinecone_api_key)
# Inicializar cliente Pinecone
index_name = "rag-literary-chatbot"

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"Index '{index_name}' already exists!")
else:
    print(f"Index '{index_name}' doesn't exist. Creating new index...")
    pc.create_index(
        name=index_name,
        dimension=1536,  # adjust based on your embedding model
        metric="cosine", 
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    

# Connect to the index
index = pc.Index(index_name)
# Inicializar Pinecone usando langchain y pasando el embedding
pinecone_vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

Index 'rag-literary-chatbot' already exists!


#### Load jsonl file

In [244]:
import json

# Load your corpus list
from pathlib import Path
path_to_read = Path("outputs") 
os.makedirs(path_to_read, exist_ok=True)
json_filename = "corpus_list.json"
jsonl_filename = "corpus_list.jsonl"
file_to_read  = os.path.join(path_to_read, json_filename)
file_to_save = os.path.join(path_to_read, jsonl_filename )


with open(file_to_read, 'r', encoding='utf-8') as f:
    corpus_list = json.load(f)

# Storage for chunks
chunk_records = []

# Configuration
MAX_TOKENS = 100  # Or characters, depending on your use case

for book in corpus_list:
    book_name = book['book_name']
    author = book['author']
    sentences_tokens = book['sentences_tokens']

    # Combine sentences into chunks
    current_chunk = []
    current_token_count = 0
    chunk_index = 0

    for sentence in sentences_tokens:
        token_count = len(sentence)

        if current_token_count + token_count <= MAX_TOKENS:
            current_chunk.extend(sentence)
            current_token_count += token_count
        else:
            # Save current chunk
            chunk_records.append({
                "chunk_text": " ".join(current_chunk),
                "metadata": {
                    "book_name": book_name,
                    "author": author,
                    "chunk_index": chunk_index
                }
            })
            chunk_index += 1

            # Start new chunk
            current_chunk = sentence.copy()
            current_token_count = token_count

    # Save any remaining chunk
    if current_chunk:
        chunk_records.append({
            "chunk_text": " ".join(current_chunk),
            "metadata": {
                "book_name": book_name,
                "author": author,
                "chunk_index": chunk_index
            }
        })

print(f"Created {len(chunk_records)} chunks total.")

# Save to JSONL
with open(file_to_save, "w", encoding="utf-8") as f:
    for record in chunk_records:
        json.dump(record, f, ensure_ascii=False)
        f.write("\n")

Created 934 chunks total.


In [245]:
import os
import json

def load_json_from_folder(folder_path: str):
    """
    Load documents from a folder containing .json or .jsonl files.
    Each doc will include text and metadata.
    """
    all_docs = []

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        lower_name = filename.lower()

        try:
            if lower_name.endswith(".jsonl"):
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        record = json.loads(line)
                        text = record.get("chunk_text", "")
                        metadata = record.get("metadata", {})
                        all_docs.append({
                            "text": text,
                            "metadata": metadata
                        })
            else:
                print(f"Skipping unsupported file: {filename}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    print(f"Loaded {len(all_docs)} documents from {folder_path}")
    return all_docs


In [246]:
from pathlib import Path
path_to_read = Path("data") / "input" / "rag"
os.makedirs(path_to_read, exist_ok=True)
jsonl_file = "corpus_list.jsonl"
json_file_to_read = os.path.join(path_to_read, jsonl_file)

# Upload jsonl file and getting the metadata and chunks
if path_to_read.exists():
    print(f"Directory exists: {path_to_read.resolve()}")
    docs_in_folder = load_json_from_folder(path_to_read)
    for doc in docs_in_folder[:2]:
        print(doc["metadata"])
        print(doc["text"][:200], "...")
    
else:
    print(f"Directory does not exist: {path_to_read}")



Directory exists: F:\IA\novelBot\data\input\rag
Loaded 934 documents from data\input\rag
{'book_name': 'pride_and_prejudice', 'author': 'Jane Austen', 'chunk_index': 0}
illustration george allen publisher charing cross road london ruskin house illustration jane letter prejudice jane austen preface george saintsbury illustrations hugh thomson illustration ruskin chari ...
{'book_name': 'pride_and_prejudice', 'author': 'Jane Austen', 'chunk_index': 1}
delightful freshness humour northanger abbey completeness finish entrain obscure undoubted critical fact small scheme burlesque parody kind first rank difficulty persuasion relatively faint tone inter ...


In [247]:
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    return client.embeddings.create(input=[text], model=model).data[0].embedding

In [248]:
# Prepare to insert vectors to Pinecone
vectors = []
for doc in docs_in_folder:
    # Generate embedding for the document text
    embedding = get_embedding(doc["text"])
    
    # Create metadata dictionary
    metadata = {
        "text": doc["text"][:100], 
        "book_name": doc["metadata"]["book_name"],
        "author": doc["metadata"]["author"],
        "chunk_index": doc["metadata"]["chunk_index"]
    }
    
    # Create a unique ID for the vector
    vector_id = f"{doc['metadata']['book_name']}_{doc['metadata']['chunk_index']}"
    
    vectors.append((vector_id, embedding, metadata))

# Upsert in batches (Pinecone recommends batches of 100)
batch_size = 16
for i in range(0, len(vectors), batch_size):
    batch = vectors[i:i+batch_size]
    index.upsert(vectors=batch)

print(f"Successfully upserted {len(vectors)} vectors to Pinecone index {index_name}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

Successfully upserted 934 vectors to Pinecone index rag-literary-chatbot


In [273]:
from typing_extensions import Annotated
class AgentState(TypedDict):
    task:str
    question: str
    plan:str
    context: Annotated[List[Document], "accumulate"]
    answer: str
    content: Annotated[List[str], "accumulate"]

In [274]:
model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [275]:
PLAN_PROMPT = """You are an expert literary analyst tasked with drafting a high-level outline for evaluating literary works, such as novels or stories.
For each text provided by the user, write this outline and include a brief summary of the main characteristics, focusing on style, tone, narrative structure, and unique literary elements.
You can read the text from the output of the load_texts_from_folder function.
Maximum 100 words."""

In [ ]:
LLM_PROMPT_LITERARY_ANALYSIS = """
You are an expert literary assistant who converts user goals into focused analytical questions to extract meaningful insights from a literary text.
Only proceed if the author's name 'Jane Austen' is explicitly mentioned in the {context}.

Generate 2 to 5 **short, specific** literary analysis questions related to evaluating the work's style, tone, narrative structure, and distinctive literary qualities.

Respond strictly in this JSON format:
{
  "queries": [
    "What are the defining elements of the author's narrative style?",
    "How does the author develop the main characters?",
    "What themes are present in the text?",
    ...
  ]
}

Call the appropriate analysis function, agent, or use the {context} to answer these questions.

Context: {context}
"""


In [ ]:
LLM_PROMPT_LITERARY_ANALYSIS_2 = """
You are an intelligent literary assistant tasked with transforming a user's goal into targeted literary analysis questions 
to help evaluate the style, narrative elements, and thematic aspects of a literary work.

Proceed only if the author's name "Jane Austen" is explicitly mentioned in the {context}. 
If "Jane Austen" is not mentioned, respond with the following JSON exactly as shown:

{
  "error": "I cannot provide the literary analysis because Jane Austen is not the author mentioned in the text."
}

If "Jane Austen" is present, generate 2 to 5 short, focused literary analysis questions that could help examine her writing style, 
character development, and narrative techniques.

Respond only in the following JSON format:
{
  "queries": [
    "What narrative techniques does Jane Austen use?",
    "How does Jane Austen develop her main characters?",
    "What themes are central to this text?",
    "How does the dialogue reflect the social context of the time?",
    "What stylistic features make this work unique?"
    "Compare the narrative between authors"
  ]
}

Context: {context}
"""


In [278]:
from pydantic import BaseModel
from typing import List

class Queries(BaseModel):
    queries: List[str]

In [292]:
def plan_node(state: AgentState):
    messages = [
        SystemMessage(content=PLAN_PROMPT), 
        HumanMessage(content=state['task'])
    ]
    response = model.invoke(messages)
    return {"plan": response.content}

In [293]:
from langchain.schema import Document
from more_itertools import chunked  # helps to split lists into batches

def retriever(state: AgentState):
    documents = [
        Document(
            page_content=chunk["chunk_text"],
            metadata=chunk["metadata"]
        )
        for chunk in chunk_records
    ]

    # Create indices for Pinecone
    indices = [
        f"{chunk['metadata']['book_name']}_{chunk['metadata']['chunk_index']}"
        for chunk in chunk_records
    ]

    # Pair each doc with its ID
    doc_pairs = list(zip(documents, indices))

    # Batch size: adjust to keep each batch under 4MB — test ~50–200 docs at a time
    BATCH_SIZE = 100

    for batch in chunked(doc_pairs, BATCH_SIZE):
        batch_docs, batch_ids = zip(*batch)
        pinecone_vectorstore.add_documents(documents=list(batch_docs), ids=list(batch_ids))
        print(f"Added batch of {len(batch_docs)} chunks to Pinecone")

    print(f"Total added: {len(documents)} chunks to Pinecone vectorstore")

    # Formulate query
    query = state.get("question") or state.get("task") or "Describe the main themes in the corpus."

    # Configure retriever
    retriever = pinecone_vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 3}
    )

    # Retrieve relevant documents
    relevant_docs = retriever.get_relevant_documents(query)
    print(f"Found {len(relevant_docs)} relevant documents for query: {query}")

    return {
        "context": relevant_docs,
        "query": query,
        "source_documents": relevant_docs
    }



In [294]:
def generate(state: AgentState):
    context_docs = state.get("context", [])
    content_chunks = state.get("content", [])

    print("Context docs:", context_docs)
    print("Content chunks:", content_chunks)

    # Combine available content
    if context_docs:
        docs_content = "\n\n".join(texts.page_content for texts in context_docs if hasattr(texts, "page_content"))
    elif content_chunks:
        docs_content = "\n\n".join([str(chunk) for chunk in content_chunks])
    else:
        return {"answer": "No context or content available for answer generation."}

    # Get a safe fallback question string
    question = {state.get('question') or state.get('task') or "Describe the main characteristics of the resumes."}  

    prompt_text = f"""Eres un asistente experto. Usa el siguiente contexto para responder a las preguntas.

Question: {question}

Contexto:
{docs_content}

Answer:"""

    try:
        response = model.invoke([HumanMessage(content=prompt_text)])
        return {"answer": response.content}
    except Exception as e:
        print("Error during model invocation:", e)
        return {"answer": f"Error invoking model: {str(e)}"}

In [295]:
def llm_user1_node(state: AgentState):
    structured_llm = model.with_structured_output(Queries)
    messages = [
        SystemMessage(content=LLM_PROMPT_LITERARY_ANALYSIS),
        HumanMessage(content=state.get('task', ''))
    ]

    try:
        queries = structured_llm.invoke(messages)
        print("Queries:", queries)
    except Exception as e:
        return {"content": [], "error": f"Failed to generate queries: {e}"}

    if not queries or not getattr(queries, "queries", None):
        return {"content": [], "error": "No queries generated"}

    llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=pinecone_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
        return_source_documents=True 
    )

    content = []
    for q in queries.queries:
        if not isinstance(q, str):
            continue
        result = qa_chain.invoke({"query": q})
        print(f"Answer: {result['result']}")
        content.append(result["result"])
       
    return {"content": content}

In [283]:
def llm_user2_node(state: AgentState):
    structured_llm = model.with_structured_output(Queries)
    messages = [
        SystemMessage(content=LLM_PROMPT_LITERARY_ANALYSIS_2),
        HumanMessage(content=state.get('task', ''))
    ]

    try:
        queries = structured_llm.invoke(messages)
        print("Queries:", queries)
    except Exception as e:
        return {"content": [], "error": f"Failed to generate queries: {e}"}

    if not queries or not getattr(queries, "queries", None):
        return {"content": [], "error": "No queries generated"}

    # Setup QA chain
    llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=pinecone_vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}),
        return_source_documents=False
    )

    content = []
    for q in queries.queries:
        if not isinstance(q, str):
            continue
        result = qa_chain.invoke({"query": q})
        print(f"Answer: {result['result']}")
        content.append(result["result"])

    return {"content": content}


In [296]:
def should_continue(state):
    return END

In [297]:
builder = StateGraph(AgentState)

In [298]:
builder.add_node("planner", plan_node)
builder.add_node("Evaluator1", llm_user1_node)
builder.add_node("Evaluator2", llm_user2_node)
builder.add_node("generate", generate)
builder.add_node("retriever", retriever)

builder.set_entry_point("planner")
builder.set_finish_point("Evaluator2")  
#builder.set_finish_point("Evaluator2")  

builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "generate")
builder.add_edge("generate", "Evaluator1")
builder.add_edge("Evaluator1", "Evaluator2")

In [299]:
builder.set_entry_point("planner")

In [300]:
builder.add_conditional_edges(
    "generate", 
    should_continue, 
    {END: END}
)

In [301]:
graph = builder.compile(checkpointer=memory)

In [302]:
from IPython.display import Image

#Image(graph.get_graph().draw_png())

In [303]:
from typing import List, Dict

# Simulación del stream y de las salidas
thread = {"configurable": {"thread_id": "1"}}

# Función para generar una salida más bonita
def print_pretty_output():
    initial_state = {
        "task": (
            "Evaluate each of the provided literary works. "
            "Summarize the main characteristics of the text, highlighting its distinctive style, themes, "
            "and narrative techniques."
        ),
        "content": []
    }

    for s in graph.stream(initial_state, thread):
        display(Markdown(f"### System answer: \n\n{s}"))

# Llamar la función
print_pretty_output()

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


### System answer: 

{'planner': {'plan': 'Sure! Please provide the texts you would like me to evaluate, and I will create a high-level outline for each, summarizing their main characteristics, style, tone, narrative structure, and unique literary elements.'}}

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Added batch of 100 chunks to Pinecone


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Answer: I don't know.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Answer: I don't know.


### System answer: 

{'Evaluator2': {'content': ['The author employs several narrative techniques in this work, including:\n\n1. **Psychological Depth**: The narrative delves into the psychological states of characters, exploring their thoughts, emotions, and inner conflicts, which adds complexity to their motivations and actions.\n\n2. **Vivid Imagery and Symbolism**: The use of vivid imagery and symbols, such as the "monstrous orchid" and "poisonous book," enhances the thematic depth and evokes strong sensory responses from the reader.\n\n3. **Stream of Consciousness**: The narrative may utilize a stream of consciousness technique, reflecting the characters\' thoughts and feelings in a flowing, often fragmented manner, which captures the complexity of their inner lives.\n\n4. **Complex Structure**: The work is structured in a way that may involve multiple chapters and shifts in perspective, allowing for a multifaceted exploration of themes and character relationships.\n\n5. **Subtle Monotony and Cadence**: The author may employ a rhythmic quality to the prose, creating a sense of musicality that reflects the emotional states of the characters and the overall mood of the narrative.\n\n6. **Irony and Contrast**: The narrative may include elements of irony, particularly in the contrast between characters\' perceptions and reality, which can highlight themes of falsehood and truth.\n\nThese techniques work together to create a rich, layered narrative that engages the reader on both emotional and intellectual levels.', "I don't know.", "The text explores several central themes and motifs, including:\n\n1. **Childhood and Innocence**: The adventures of Tom Sawyer highlight the joys and challenges of childhood, emphasizing the innocence and curiosity of youth.\n\n2. **Rebellion and Freedom**: Tom's actions often reflect a desire for freedom and rebellion against societal norms and expectations, showcasing the struggle between individual desires and social responsibilities.\n\n3. **Moral Development**: The narrative examines Tom's moral growth as he navigates various challenges and temptations, reflecting on duty, guilt, and the consequences of one's actions.\n\n4. **Friendship and Loyalty**: The relationships between characters, particularly Tom and his friends, underscore the importance of friendship and loyalty in the face of adversity.\n\n5. **Society and Class**: The text also touches on social class and the dynamics of society, particularly through the interactions between different characters and their backgrounds.\n\n6. **Adventure and Imagination**: The motif of adventure is prevalent, as Tom's escapades often blur the lines between reality and imagination, highlighting the power of creativity and storytelling.\n\nThese themes and motifs contribute to the rich psychological and social commentary present in the narrative.", "I don't know.", "I don't know.", "I don't know."]}}